# Fraud Detection Data Analysis System
### BDA Project | PaySim Financial Dataset
---
**Dataset:** PS_20174392719_1491204439457_log.csv  
**Total Records:** 6,362,620  
**Total Columns:** 11  

**Phases:**
- Phase 1: Load & Explore Dataset
- Phase 2: Data Cleaning
- Phase 3: Outlier Detection (Z-Score)
- Phase 4: User Grouping & Spending Behavior
- Phase 5: Fraud Risk Score (Feature Engineering)
- Phase 6: Summary Statistics & Correlation Analysis

In [ ]:
# ENVIRONMENT SETUP
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 1000)

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully!")
print(f"   Pandas  version : {pd.__version__}")
print(f"   NumPy   version : {np.__version__}")
print(f"   Seaborn version : {sns.__version__}")
print("\nEnvironment is ready. Proceeding to Phase 1...")

---
## Phase 1: Load & Explore Dataset
**Goal:** Load the PaySim dataset and get a complete overview of its structure, contents, and basic statistics.

In [ ]:
# PHASE 1 — LOAD & EXPLORE DATASET
path = r"C:\Users\hp\Desktop\bda proj\PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(path)
print("Dataset Loaded Successfully!")

print(f"\nShape of Dataset:")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")

print(f"\nColumn Names:")
print(df.columns.tolist())

print(f"\nData Types:")
print(df.dtypes)

print(f"\nFirst 5 Rows:")
print(df.head())

print(f"\nMissing Values Per Column:")
print(df.isnull().sum())

print(f"\nDuplicate Rows: {df.duplicated().sum()}")

print(f"\nFraud Label Distribution:")
print(df['isFraud'].value_counts())
print(f"\n   Fraud Percentage:")
print(round(df['isFraud'].value_counts(normalize=True) * 100, 4))

print(f"\nTransaction Type Counts:")
print(df['type'].value_counts())

print(f"\nBasic Statistics:")
print(df.describe())

In [ ]:
import os
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.facecolor' : '#0a0a0a',
    'axes.facecolor'   : '#0f0f1a',
    'axes.edgecolor'   : '#00ffff',
    'axes.labelcolor'  : '#00ffff',
    'axes.titlecolor'  : '#ffffff',
    'xtick.color'      : '#00ffff',
    'ytick.color'      : '#00ffff',
    'text.color'       : '#ffffff',
    'grid.color'       : '#1a1a2e',
    'grid.linestyle'   : '--',
    'axes.grid'        : True,
    'font.family'      : 'monospace',
})

BG      = '#0a0a0a'
PANEL   = '#0f0f1a'
CYAN    = '#00ffff'
GREEN   = '#00ff88'
RED     = '#ff003c'
MAGENTA = '#ff00ff'
ORANGE  = '#ff6600'
PURPLE  = '#7b2fff'
OUTPUT  = r"C:\Users\hp\Desktop\bda proj\outputs"
os.makedirs(OUTPUT, exist_ok=True)

# PHASE 1 — VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('[ PHASE 1 — DATASET OVERVIEW ]', fontsize=14,
             color=CYAN, fontfamily='monospace', fontweight='bold')

# Before: Raw transaction type counts
ax = axes[0]
ax.set_facecolor(PANEL)
type_counts = df['type'].value_counts()
colors = [CYAN, GREEN, MAGENTA, ORANGE, PURPLE]
bars = ax.bar(type_counts.index, type_counts.values, color=colors, edgecolor=BG, width=0.5)
for bar, color in zip(bars, colors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15000,
            f'{int(bar.get_height()):,}', ha='center', color=color, fontsize=8, fontweight='bold')
ax.set_title('BEFORE — Raw Transaction Types', color=CYAN, fontsize=11)
ax.set_xlabel('Type', color=CYAN); ax.set_ylabel('Count', color=CYAN)
ax.spines[:].set_color(CYAN)

# After: Fraud vs Non-Fraud breakdown
ax = axes[1]
ax.set_facecolor(PANEL)
fraud_counts = df['isFraud'].value_counts()
labels = ['Non-Fraud', 'Fraud']
values = [fraud_counts[0], fraud_counts[1]]
b = ax.bar(labels, values, color=[GREEN, RED], edgecolor=BG, width=0.4)
for bar, val, color in zip(b, values, [GREEN, RED]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
            f'{val:,}', ha='center', color=color, fontsize=9, fontweight='bold')
ax.set_title('AFTER — Fraud vs Non-Fraud Distribution', color=CYAN, fontsize=11)
ax.set_ylabel('Count', color=CYAN)
ax.spines[:].set_color(CYAN)

plt.tight_layout()
plt.savefig(f"{OUTPUT}\\phase1_overview.png", dpi=150, facecolor=BG)
plt.show()
print("Phase 1 graph saved!")

---
## Phase 2: Data Cleaning & Handling Incorrect Data
**Goal:** Identify and handle incorrect balance entries, inconsistent data, and prepare a clean version of the dataset for analysis.

In [ ]:
# PHASE 2 — DATA CLEANING
df['balance_diff_orig'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
df['balance_diff_dest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

print("Balance Inconsistency Check:")
print(f"   Sender balance mismatches   : {(df['balance_diff_orig'].abs() > 1).sum():,}")
print(f"   Receiver balance mismatches : {(df['balance_diff_dest'].abs() > 1).sum():,}")

df['account_drained'] = (
    (df['oldbalanceOrg'] > 0) &
    (df['newbalanceOrig'] == 0)
).astype(int)

print(f"\nAccounts Drained (balance went to 0):")
print(f"   Total : {df['account_drained'].sum():,}")

df['dest_not_updated'] = (
    (df['amount'] > 0) &
    (df['newbalanceDest'] == 0)
).astype(int)

print(f"\nDestination Balance Not Updated:")
print(f"   Total : {df['dest_not_updated'].sum():,}")

print(f"\nTransaction Types:")
print(df['type'].value_counts())

print(f"\nFraud Count by Transaction Type:")
print(df.groupby('type')['isFraud'].sum())

df.drop(columns=['balance_diff_orig', 'balance_diff_dest'], inplace=True)

print(f"\nCleaned Dataset Summary:")
print(f"   Total Rows       : {len(df):,}")
print(f"   Total Columns    : {len(df.columns)}")
print(f"   Account Drained  : {df['account_drained'].sum():,}")
print(f"   Dest Not Updated : {df['dest_not_updated'].sum():,}")
print(f"\nUpdated Columns:")
print(df.columns.tolist())

In [ ]:
import os
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.facecolor' : '#0a0a0a',
    'axes.facecolor'   : '#0f0f1a',
    'axes.edgecolor'   : '#00ffff',
    'axes.labelcolor'  : '#00ffff',
    'axes.titlecolor'  : '#ffffff',
    'xtick.color'      : '#00ffff',
    'ytick.color'      : '#00ffff',
    'text.color'       : '#ffffff',
    'grid.color'       : '#1a1a2e',
    'grid.linestyle'   : '--',
    'axes.grid'        : True,
    'font.family'      : 'monospace',
})

BG      = '#0a0a0a'
PANEL   = '#0f0f1a'
CYAN    = '#00ffff'
GREEN   = '#00ff88'
RED     = '#ff003c'
MAGENTA = '#ff00ff'
ORANGE  = '#ff6600'
PURPLE  = '#7b2fff'
OUTPUT  = r"C:\Users\hp\Desktop\bda proj\outputs"
os.makedirs(OUTPUT, exist_ok=True)

# PHASE 2 — VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('[ PHASE 2 — DATA CLEANING RESULTS ]', fontsize=14,
             color=CYAN, fontfamily='monospace', fontweight='bold')

# Before: All transactions unlabelled
ax = axes[0]
ax.set_facecolor(PANEL)
total = len(df)
clean = total - df['account_drained'].sum() - df['dest_not_updated'].sum()
sizes_before = [total]
ax.bar(['All Transactions'], [total], color=CYAN, edgecolor=BG, width=0.4)
ax.set_title('BEFORE — All Transactions (Unchecked)', color=CYAN, fontsize=11)
ax.set_ylabel('Count', color=CYAN)
ax.text(0, total + 50000, f'{total:,}', ha='center', color=CYAN, fontweight='bold')
ax.spines[:].set_color(CYAN)

# After: Breakdown of flagged anomalies
ax = axes[1]
ax.set_facecolor(PANEL)
labels  = ['Account\nDrained', 'Dest Not\nUpdated', 'Clean']
values  = [df['account_drained'].sum(), df['dest_not_updated'].sum(),
           total - df['account_drained'].sum() - df['dest_not_updated'].sum()]
colors  = [RED, ORANGE, GREEN]
bars = ax.bar(labels, values, color=colors, edgecolor=BG, width=0.5)
for bar, val, color in zip(bars, values, colors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
            f'{val:,}', ha='center', color=color, fontsize=8, fontweight='bold')
ax.set_title('AFTER — Anomalies Flagged', color=CYAN, fontsize=11)
ax.set_ylabel('Count', color=CYAN)
ax.spines[:].set_color(CYAN)

plt.tight_layout()
plt.savefig(f"{OUTPUT}\\phase2_cleaning.png", dpi=150, facecolor=BG)
plt.show()
print("Phase 2 graph saved!")

---
## Phase 3: Outlier Detection Using NumPy Z-Score
**Goal:** Detect unusual/suspicious transactions using statistical z-score method on the amount column and analyze how many outliers are actually fraudulent.

In [ ]:
# PHASE 3 — OUTLIER DETECTION USING Z-SCORE
mean_amount = np.mean(df['amount'])
std_amount  = np.std(df['amount'])

df['z_score'] = (df['amount'] - mean_amount) / std_amount

print("Z-Score Computed on Amount Column:")
print(f"   Mean Amount : {mean_amount:,.2f}")
print(f"   Std  Amount : {std_amount:,.2f}")

df['is_outlier'] = (df['z_score'].abs() > 3).astype(int)

total_outliers = df['is_outlier'].sum()
print(f"\nOutliers Detected (Z-Score > 3):")
print(f"   Total Outliers : {total_outliers:,}")
print(f"   Percentage     : {round(total_outliers / len(df) * 100, 2)}%")

print(f"\nOutliers by Transaction Type:")
print(df[df['is_outlier'] == 1]['type'].value_counts())

outlier_fraud = df[(df['is_outlier'] == 1) & (df['isFraud'] == 1)]
print(f"\nOutliers That Are Fraud:")
print(f"   Total          : {len(outlier_fraud):,}")
print(f"   Out of Total Fraud ({df['isFraud'].sum():,}) : {round(len(outlier_fraud)/df['isFraud'].sum()*100,2)}%")

print(f"\nOutlier Amount Statistics:")
print(df[df['is_outlier'] == 1]['amount'].describe())

print(f"\nFraud Rate Comparison:")
print(f"   Fraud rate in NON-outliers : {round(df[df['is_outlier']==0]['isFraud'].mean()*100,4)}%")
print(f"   Fraud rate in OUTLIERS     : {round(df[df['is_outlier']==1]['isFraud'].mean()*100,4)}%")

print(f"\nTop 10 Highest Z-Score Transactions:")
print(df[['type','amount','z_score','isFraud']].sort_values('z_score', ascending=False).head(10))

print(f"\nPhase 3 Complete!")
print(f"   New Column Added : z_score")
print(f"   New Column Added : is_outlier")

In [ ]:
import os
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.facecolor' : '#0a0a0a',
    'axes.facecolor'   : '#0f0f1a',
    'axes.edgecolor'   : '#00ffff',
    'axes.labelcolor'  : '#00ffff',
    'axes.titlecolor'  : '#ffffff',
    'xtick.color'      : '#00ffff',
    'ytick.color'      : '#00ffff',
    'text.color'       : '#ffffff',
    'grid.color'       : '#1a1a2e',
    'grid.linestyle'   : '--',
    'axes.grid'        : True,
    'font.family'      : 'monospace',
})

BG      = '#0a0a0a'
PANEL   = '#0f0f1a'
CYAN    = '#00ffff'
GREEN   = '#00ff88'
RED     = '#ff003c'
MAGENTA = '#ff00ff'
ORANGE  = '#ff6600'
PURPLE  = '#7b2fff'
OUTPUT  = r"C:\Users\hp\Desktop\bda proj\outputs"
os.makedirs(OUTPUT, exist_ok=True)

# PHASE 3 — VISUALIZATION
sample = df.sample(5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('[ PHASE 3 — OUTLIER DETECTION (Z-SCORE) ]', fontsize=14,
             color=CYAN, fontfamily='monospace', fontweight='bold')

# Before: Raw amount distribution — no outlier labels
ax = axes[0]
ax.set_facecolor(PANEL)
ax.hist(sample['amount'], bins=40, color=CYAN, edgecolor=BG, alpha=0.85)
ax.set_title('BEFORE — Raw Amount Distribution', color=CYAN, fontsize=11)
ax.set_xlabel('Transaction Amount', color=CYAN)
ax.set_ylabel('Frequency', color=CYAN)
ax.spines[:].set_color(CYAN)

# After: Scatter with outliers highlighted and threshold line
ax = axes[1]
ax.set_facecolor(PANEL)
outlier_colors = sample['is_outlier'].map({0: CYAN, 1: ORANGE})
ax.scatter(sample['amount'], sample['z_score'],
           c=outlier_colors, alpha=0.5, s=12, edgecolors='none')
ax.axhline(y=3,  color=RED, linestyle='--', linewidth=1.2)
ax.axhline(y=-3, color=RED, linestyle='--', linewidth=1.2)
ax.set_title('AFTER — Z-Score with Outliers Flagged', color=CYAN, fontsize=11)
ax.set_xlabel('Transaction Amount', color=CYAN)
ax.set_ylabel('Z-Score', color=CYAN)
legend = [Patch(color=CYAN, label='Normal'), Patch(color=ORANGE, label='Outlier'),
          Patch(color=RED, label='Threshold (z=3)')]
ax.legend(handles=legend, facecolor=PANEL, edgecolor=CYAN, labelcolor='white', fontsize=8)
ax.spines[:].set_color(CYAN)

plt.tight_layout()
plt.savefig(f"{OUTPUT}\\phase3_outliers.png", dpi=150, facecolor=BG)
plt.show()
print("Phase 3 graph saved!")

---
## Phase 4: User Grouping & Spending Behavior Analysis
**Goal:** Group transactions by each user, analyze their spending patterns, and identify high-risk users based on their transaction behavior.

In [ ]:
# PHASE 4 — USER GROUPING & SPENDING BEHAVIOR
user_group = df.groupby('nameOrig').agg(
    total_transactions = ('amount',  'count'),
    total_spent        = ('amount',  'sum'),
    avg_transaction    = ('amount',  'mean'),
    max_transaction    = ('amount',  'max'),
    min_transaction    = ('amount',  'min'),
    total_fraud        = ('isFraud', 'sum'),
    total_drained      = ('account_drained', 'sum'),
    total_outliers     = ('is_outlier', 'sum')
).reset_index()

print("User Grouping Complete!")
print(f"   Total Unique Users : {len(user_group):,}")
print(f"\nUser Group Sample (First 5):")
print(user_group.head())

print(f"\nSpending Behavior Statistics:")
print(f"   Avg transactions per user : {round(user_group['total_transactions'].mean(),2)}")
print(f"   Avg total spent per user  : {round(user_group['total_spent'].mean(),2):,}")
print(f"   Max single transaction    : {user_group['max_transaction'].max():,}")

high_risk = user_group[
    (user_group['total_fraud']    > 0) |
    (user_group['total_outliers'] > 0) |
    (user_group['total_drained']  > 0)
]

print(f"\nHigh Risk Users Identified:")
print(f"   Total High Risk Users : {len(high_risk):,}")
print(f"   Out of Total Users    : {len(user_group):,}")
print(f"   Percentage            : {round(len(high_risk)/len(user_group)*100,2)}%")

print(f"\nTop 10 Users by Total Amount Spent:")
print(user_group.sort_values('total_spent', ascending=False)[
    ['nameOrig','total_transactions','total_spent','avg_transaction','total_fraud']
].head(10))

fraud_users = user_group[user_group['total_fraud'] > 0]
print(f"\nUsers With Fraud Transactions: {len(fraud_users):,}")

user_group['spending_category'] = pd.cut(
    user_group['total_spent'],
    bins   = [0, 100000, 500000, 1000000, float('inf')],
    labels = ['Low', 'Medium', 'High', 'Very High']
)

print(f"\nUsers by Spending Category:")
print(user_group['spending_category'].value_counts())
print(f"\nPhase 4 Complete! user_group DataFrame created with {len(user_group):,} users")

In [ ]:
import os
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.facecolor' : '#0a0a0a',
    'axes.facecolor'   : '#0f0f1a',
    'axes.edgecolor'   : '#00ffff',
    'axes.labelcolor'  : '#00ffff',
    'axes.titlecolor'  : '#ffffff',
    'xtick.color'      : '#00ffff',
    'ytick.color'      : '#00ffff',
    'text.color'       : '#ffffff',
    'grid.color'       : '#1a1a2e',
    'grid.linestyle'   : '--',
    'axes.grid'        : True,
    'font.family'      : 'monospace',
})

BG      = '#0a0a0a'
PANEL   = '#0f0f1a'
CYAN    = '#00ffff'
GREEN   = '#00ff88'
RED     = '#ff003c'
MAGENTA = '#ff00ff'
ORANGE  = '#ff6600'
PURPLE  = '#7b2fff'
OUTPUT  = r"C:\Users\hp\Desktop\bda proj\outputs"
os.makedirs(OUTPUT, exist_ok=True)

# PHASE 4 — VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('[ PHASE 4 — USER GROUPING & RISK PROFILE ]', fontsize=14,
             color=CYAN, fontfamily='monospace', fontweight='bold')

# Before: Raw transaction amounts — no user context
ax = axes[0]
ax.set_facecolor(PANEL)
sample_raw = df['amount'].sample(5000, random_state=1)
ax.hist(sample_raw, bins=40, color=CYAN, edgecolor=BG, alpha=0.85)
ax.set_title('BEFORE — Transaction Amounts (No User Context)', color=CYAN, fontsize=11)
ax.set_xlabel('Amount', color=CYAN); ax.set_ylabel('Frequency', color=CYAN)
ax.spines[:].set_color(CYAN)

# After: Spending category distribution of users
ax = axes[1]
ax.set_facecolor(PANEL)
cat_counts = user_group['spending_category'].value_counts()
cat_colors = [GREEN, CYAN, ORANGE, RED]
bars = ax.bar(cat_counts.index, cat_counts.values, color=cat_colors, edgecolor=BG, width=0.5)
for bar, val, color in zip(bars, cat_counts.values, cat_colors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,
            f'{val:,}', ha='center', color=color, fontsize=8, fontweight='bold')
ax.set_title('AFTER — Users by Spending Category', color=CYAN, fontsize=11)
ax.set_xlabel('Spending Category', color=CYAN); ax.set_ylabel('Number of Users', color=CYAN)
ax.spines[:].set_color(CYAN)

plt.tight_layout()
plt.savefig(f"{OUTPUT}\\phase4_user_grouping.png", dpi=150, facecolor=BG)
plt.show()
print("Phase 4 graph saved!")

---
## Phase 5: Fraud Risk Score (Feature Engineering)
**Goal:** Build a fraud risk score (0-100) for every transaction by combining all signals discovered in previous phases and categorize each transaction by risk level.

In [ ]:
# PHASE 5 — FRAUD RISK SCORE (FEATURE ENGINEERING)
df['risk_type']    = df['type'].apply(lambda x: 25 if x in ['TRANSFER', 'CASH_OUT'] else 0)
df['risk_outlier'] = df['is_outlier'] * 25
df['risk_drained'] = df['account_drained'] * 20
df['risk_dest']    = df['dest_not_updated'] * 20
df['risk_flagged'] = df['isFlaggedFraud'] * 10

print("Risk Signals Created:")
print(f"   Signal 1 - Risky Type (25pts)       : {(df['risk_type']>0).sum():,} transactions")
print(f"   Signal 2 - Outlier Amount (25pts)   : {(df['risk_outlier']>0).sum():,} transactions")
print(f"   Signal 3 - Account Drained (20pts)  : {(df['risk_drained']>0).sum():,} transactions")
print(f"   Signal 4 - Dest Not Updated (20pts) : {(df['risk_dest']>0).sum():,} transactions")
print(f"   Signal 5 - System Flagged (10pts)   : {(df['risk_flagged']>0).sum():,} transactions")

df['fraud_risk_score'] = (df['risk_type'] + df['risk_outlier'] +
                           df['risk_drained'] + df['risk_dest'] + df['risk_flagged'])

print(f"\nFraud Risk Score Statistics:")
print(df['fraud_risk_score'].describe())

df['risk_level'] = pd.cut(
    df['fraud_risk_score'],
    bins   = [-1, 20, 45, 70, 100],
    labels = ['Low', 'Medium', 'High', 'Critical']
)

print(f"\nRisk Level Distribution:")
print(df['risk_level'].value_counts())
print(f"\n   Percentage:")
print(round(df['risk_level'].value_counts(normalize=True) * 100, 2))

print(f"\nActual Fraud Count by Risk Level:")
print(df.groupby('risk_level', observed=True)['isFraud'].sum())

print(f"\nFraud Rate by Risk Level:")
print(round(df.groupby('risk_level', observed=True)['isFraud'].mean() * 100, 4))

critical = df[df['risk_level'] == 'Critical']
high     = df[df['risk_level'] == 'High']
print(f"\nCritical Risk Transactions : {len(critical):,}")
print(f"High Risk Transactions     : {len(high):,}")

df.drop(columns=['risk_type','risk_outlier','risk_drained','risk_dest','risk_flagged'], inplace=True)

print(f"\nPhase 5 Complete!")
print(f"   New Column Added : fraud_risk_score (0-100)")
print(f"   New Column Added : risk_level (Low/Medium/High/Critical)")
print(f"\nFinal df Columns:")
print(df.columns.tolist())

In [ ]:
import os
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.facecolor' : '#0a0a0a',
    'axes.facecolor'   : '#0f0f1a',
    'axes.edgecolor'   : '#00ffff',
    'axes.labelcolor'  : '#00ffff',
    'axes.titlecolor'  : '#ffffff',
    'xtick.color'      : '#00ffff',
    'ytick.color'      : '#00ffff',
    'text.color'       : '#ffffff',
    'grid.color'       : '#1a1a2e',
    'grid.linestyle'   : '--',
    'axes.grid'        : True,
    'font.family'      : 'monospace',
})

BG      = '#0a0a0a'
PANEL   = '#0f0f1a'
CYAN    = '#00ffff'
GREEN   = '#00ff88'
RED     = '#ff003c'
MAGENTA = '#ff00ff'
ORANGE  = '#ff6600'
PURPLE  = '#7b2fff'
OUTPUT  = r"C:\Users\hp\Desktop\bda proj\outputs"
os.makedirs(OUTPUT, exist_ok=True)

# PHASE 5 — VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('[ PHASE 5 — FRAUD RISK SCORE ]', fontsize=14,
             color=CYAN, fontfamily='monospace', fontweight='bold')

# Before: Plain isFraud column — only 0 or 1, no nuance
ax = axes[0]
ax.set_facecolor(PANEL)
raw_fraud = df['isFraud'].value_counts()
ax.bar(['Non-Fraud (0)', 'Fraud (1)'], [raw_fraud[0], raw_fraud[1]],
       color=[CYAN, RED], edgecolor=BG, width=0.4)
ax.set_title('BEFORE — Only Binary Fraud Label', color=CYAN, fontsize=11)
ax.set_ylabel('Count', color=CYAN)
ax.spines[:].set_color(CYAN)

# After: Full risk score spectrum
ax = axes[1]
ax.set_facecolor(PANEL)
n, bins, patches = ax.hist(df['fraud_risk_score'], bins=20, edgecolor=BG)
for patch, left in zip(patches, bins[:-1]):
    if left < 25:   patch.set_facecolor(GREEN)
    elif left < 50: patch.set_facecolor(ORANGE)
    elif left < 75: patch.set_facecolor(MAGENTA)
    else:           patch.set_facecolor(RED)
ax.set_title('AFTER — Risk Score Distribution (0-100)', color=CYAN, fontsize=11)
ax.set_xlabel('Risk Score', color=CYAN); ax.set_ylabel('Transactions', color=CYAN)
legend = [Patch(color=GREEN, label='Low'), Patch(color=ORANGE, label='Medium'),
          Patch(color=MAGENTA, label='High'), Patch(color=RED, label='Critical')]
ax.legend(handles=legend, facecolor=PANEL, edgecolor=CYAN, labelcolor='white', fontsize=8)
ax.spines[:].set_color(CYAN)

plt.tight_layout()
plt.savefig(f"{OUTPUT}\\phase5_risk_score.png", dpi=150, facecolor=BG)
plt.show()
print("Phase 5 graph saved!")

---
## Phase 6: Summary Statistics & Correlation Analysis
**Goal:** Generate complete summary statistics, correlation analysis, and visualizations for the entire project.

In [ ]:
# PHASE 6 — SUMMARY STATISTICS & CORRELATION ANALYSIS
import os
os.makedirs(r"C:\Users\hp\Desktop\bda proj\outputs", exist_ok=True)

numeric_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig',
                'oldbalanceDest', 'newbalanceDest',
                'fraud_risk_score', 'z_score']

print("=" * 60)
print("COMPLETE DATASET SUMMARY STATISTICS")
print("=" * 60)
print(df[numeric_cols].describe().round(2))

fraud     = df[df['isFraud'] == 1]
non_fraud = df[df['isFraud'] == 0]

print("\n" + "=" * 60)
print("FRAUD vs NON-FRAUD TRANSACTION STATISTICS")
print("=" * 60)
print(f"\n{'Metric':<30} {'Fraud':>15} {'Non-Fraud':>15}")
print("-" * 60)
print(f"{'Total Transactions':<30} {len(fraud):>15,} {len(non_fraud):>15,}")
print(f"{'Mean Amount':<30} {fraud['amount'].mean():>15,.2f} {non_fraud['amount'].mean():>15,.2f}")
print(f"{'Max Amount':<30} {fraud['amount'].max():>15,.2f} {non_fraud['amount'].max():>15,.2f}")
print(f"{'Mean Risk Score':<30} {fraud['fraud_risk_score'].mean():>15,.2f} {non_fraud['fraud_risk_score'].mean():>15,.2f}")
print(f"{'Outliers Count':<30} {fraud['is_outlier'].sum():>15,} {non_fraud['is_outlier'].sum():>15,}")
print(f"{'Account Drained':<30} {fraud['account_drained'].sum():>15,} {non_fraud['account_drained'].sum():>15,}")
print(f"{'Dest Not Updated':<30} {fraud['dest_not_updated'].sum():>15,} {non_fraud['dest_not_updated'].sum():>15,}")

corr_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
             'newbalanceDest', 'isFraud', 'account_drained', 'dest_not_updated',
             'z_score', 'is_outlier', 'fraud_risk_score']
corr_matrix = df[corr_cols].corr().round(3)

print("\n" + "=" * 60)
print("CORRELATION WITH isFraud (sorted):")
print("=" * 60)
print(corr_matrix['isFraud'].sort_values(ascending=False))

summary_df = pd.DataFrame({
    'Phase': ['Phase 1','Phase 2','Phase 2','Phase 3','Phase 3',
              'Phase 4','Phase 4','Phase 5','Phase 5'],
    'What We Found': [
        'Dataset loaded & explored', 'Account drained anomalies',
        'Destination not updated', 'Outliers detected (z>3)',
        'Outliers that are fraud', 'Unique users identified',
        'High risk users found', 'Risk score computed', 'Critical risk transactions'
    ],
    'Count': [
        f"{len(df):,}", f"{df['account_drained'].sum():,}",
        f"{df['dest_not_updated'].sum():,}", f"{df['is_outlier'].sum():,}",
        f"{((df['is_outlier']==1) & (df['isFraud']==1)).sum():,}",
        f"{len(user_group):,}",
        f"{len(user_group[(user_group['total_fraud']>0)|(user_group['total_outliers']>0)|(user_group['total_drained']>0)]):,}",
        "0 - 100 scale", f"{(df['risk_level']=='Critical').sum():,}"
    ]
})

print("\n" + "=" * 60)
print("PROJECT IMPROVEMENTS SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

In [ ]:
import os
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.facecolor' : '#0a0a0a',
    'axes.facecolor'   : '#0f0f1a',
    'axes.edgecolor'   : '#00ffff',
    'axes.labelcolor'  : '#00ffff',
    'axes.titlecolor'  : '#ffffff',
    'xtick.color'      : '#00ffff',
    'ytick.color'      : '#00ffff',
    'text.color'       : '#ffffff',
    'grid.color'       : '#1a1a2e',
    'grid.linestyle'   : '--',
    'axes.grid'        : True,
    'font.family'      : 'monospace',
})

BG      = '#0a0a0a'
PANEL   = '#0f0f1a'
CYAN    = '#00ffff'
GREEN   = '#00ff88'
RED     = '#ff003c'
MAGENTA = '#ff00ff'
ORANGE  = '#ff6600'
PURPLE  = '#7b2fff'
OUTPUT  = r"C:\Users\hp\Desktop\bda proj\outputs"
os.makedirs(OUTPUT, exist_ok=True)

# PHASE 6 — VISUALIZATION

corr_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
             'newbalanceDest', 'isFraud', 'account_drained', 'dest_not_updated',
             'z_score', 'is_outlier', 'fraud_risk_score']
corr_matrix = df[corr_cols].corr().round(3)
fraud     = df[df['isFraud'] == 1]
non_fraud = df[df['isFraud'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
fig.suptitle('[ PHASE 6 — CORRELATION & FRAUD COMPARISON ]', fontsize=14,
             color=CYAN, fontfamily='monospace', fontweight='bold')

# Before: Unweighted — all features equal, no correlation insight
ax = axes[0]
ax.set_facecolor(PANEL)
key_cols   = ['amount', 'isFraud', 'account_drained',
              'dest_not_updated', 'z_score', 'is_outlier', 'fraud_risk_score']
small_corr = df[key_cols].corr().round(2)
sns.heatmap(small_corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.8, linecolor=BG,
            annot_kws={'size': 8, 'color': 'white'}, ax=ax,
            cbar_kws={'shrink': 0.75})
ax.set_title('BEFORE — Feature Correlation Heatmap', color=CYAN, fontsize=11)
ax.tick_params(colors=CYAN, labelsize=7)

# After: Scatter — amount vs risk score, fraud highlighted
ax = axes[1]
ax.set_facecolor(PANEL)
sample = df.sample(5000, random_state=42)
sc = sample['isFraud'].map({0: CYAN, 1: RED})
ax.scatter(sample['amount'], sample['fraud_risk_score'],
           c=sc, alpha=0.5, s=12, edgecolors='none')
ax.set_title('AFTER — Amount vs Risk Score (Fraud Highlighted)', color=CYAN, fontsize=11)
ax.set_xlabel('Transaction Amount', color=CYAN)
ax.set_ylabel('Fraud Risk Score', color=CYAN)
legend = [Patch(color=CYAN, label='Non-Fraud'), Patch(color=RED, label='Fraud')]
ax.legend(handles=legend, facecolor=PANEL, edgecolor=CYAN, labelcolor='white', fontsize=8)
ax.spines[:].set_color(CYAN)

plt.tight_layout()
plt.savefig(f"{OUTPUT}\\phase6_summary.png", dpi=150, facecolor=BG)
plt.show()
print("Phase 6 graph saved!")

print("\n" + "=" * 60)
print("PROJECT COMPLETE — ALL 6 PHASES DONE!")
print("=" * 60)
print(f"   Total Transactions : {len(df):,}")
print(f"   Fraud Detected     : {df['isFraud'].sum():,}")
print(f"   Outliers Found     : {df['is_outlier'].sum():,}")
print(f"   Graphs Saved       : {OUTPUT}")
print("=" * 60)